# Middleware

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

## 1. Wrap-style Hooks

### 1) wrap_model_call

In [2]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage

In [3]:
basic_model = init_chat_model(model='openai:gpt-5.4-nano')
advanced_model = init_chat_model(model='openai:gpt-5.4-mini')

In [6]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    
    print('middleware 실행 : dynamic_model_selection')
    
    message_len = len(request.state['messages'])
    
    if message_len > 10:
        model = advanced_model
    else:
        model = basic_model
        
    return handler(request.override(model = model))

In [7]:
agent = create_agent(
    model=basic_model,
    tools=[],
    middleware=[dynamic_model_selection]
)

In [8]:
agent.invoke(
    {
        'messages': HumanMessage(content='RAG가 무엇인지 설명해주세요.')
    }
)

middleware 실행 : dynamic_model_selection


{'messages': [HumanMessage(content='RAG가 무엇인지 설명해주세요.', additional_kwargs={}, response_metadata={}, id='9a098868-8b8d-44db-b078-8ba9f0f7f998'),
  AIMessage(content='RAG는 **Retrieval-Augmented Generation**의 약자로, 생성형 AI(예: LLM)가 **필요한 정보를 검색(검색/조회)해서 가져온 뒤**, 그 정보를 바탕으로 답변을 생성하는 방식을 말합니다.\n\n핵심은 두 단계입니다:\n\n1. **Retrieval(검색/조회)**  \n   - 사용자의 질문에 관련된 문서(지식) 조각을 외부에서 찾아옵니다.  \n   - 보통은 사내 문서, 위키, DB, PDF 등을 **임베딩(벡터화)** 해두고, 질문과의 유사도를 계산해 관련 내용을 가져옵니다.\n\n2. **Generation(생성)**  \n   - 검색된 내용(컨텍스트)을 프롬프트에 넣고, LLM이 그 근거를 바탕으로 답을 작성합니다.\n\n### 왜 RAG를 쓰나요?\n- **환각(없는 내용을 지어내는 문제)**을 줄이는 데 도움이 됩니다.  \n- 최신/사내 데이터처럼 **정형 지식 업데이트**가 필요한 경우에 유리합니다.  \n- LLM 자체의 학습 데이터에 없는 정보도 외부 지식으로 커버할 수 있습니다.\n\n원하시면 “간단한 구성도(데이터 흐름)”나 “구현 예시(벡터DB+검색+프롬프트 구성)”도 같이 설명해드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 307, 'prompt_tokens': 14, 'total_tokens': 321, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_token

### 2) wrap_tool_call

In [10]:
from dataclasses import dataclass
from typing import Any

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langchain_community.tools import DuckDuckGoSearchResults

In [ ]:
from collections.abc import Callable

from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest
from langgraph.types import Command


@wrap_tool_call
def log_tool_calls(
    request: ToolCallRequest,                                    # request: Agent가 갖고있는 정보들
    handler: Callable[[ToolCallRequest], ToolMessage | Command], # handler: Agent == "다음 실행 단계"
) -> ToolMessage | Command:
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]

    print(f"[TOOL START]{tool_name}/args={tool_args}")

    try:
        result = handler(request)

        if isinstance(result, ToolMessage):
            print(f"[TOOL END]{tool_name}/result={result.content}")
        else:
            print(f"[TOOL END]{tool_name}/result={result}")

        return result

    except Exception as e:
        print(f"[TOOL ERROR]{tool_name}/error={e}")
        raise

In [15]:
# -----------------------------------
# 1. 사용자 context
# -----------------------------------
@dataclass
class UserContext:
    user_id: str


# -----------------------------------
# 2. 장기 메모리 저장소
# -----------------------------------
store = InMemoryStore()


# -----------------------------------
# 3. DuckDuckGo 검색 도구 준비
# -----------------------------------
# output_format="list" 로 받으면 검색 결과를 구조화된 형태로 다루기 쉽습니다.
ddg_tool = DuckDuckGoSearchResults(
    output_format="list",
    max_results=8,
)


# -----------------------------------
# 4. 좋아하는 영화 장르 저장 tool
# -----------------------------------
@tool
def save_favorite_genre(genre: str, runtime: ToolRuntime[UserContext]) -> str:
    """
    사용자가 좋아하는 영화 장르를 장기 메모리에 저장합니다.
    예: genre='sci-fi'
    """
    assert runtime.store is not None

    namespace = ("users", runtime.context.user_id, "movie_preferences")

    runtime.store.put(
        namespace,
        "favorite_genre",
        {"genre": genre}
    )

    return f"좋아하는 영화 장르를 '{genre}'로 저장했습니다."


# -----------------------------------
# 5. 저장된 장르 조회 tool
# -----------------------------------
@tool
def load_favorite_genre(runtime: ToolRuntime[UserContext]) -> str:
    """
    저장된 영화 장르를 조회합니다.
    """
    assert runtime.store is not None

    namespace = ("users", runtime.context.user_id, "movie_preferences")
    memory = runtime.store.get(namespace, "favorite_genre")

    if memory is None:
        return "아직 저장된 좋아하는 영화 장르가 없습니다."

    return f"저장된 좋아하는 영화 장르는 '{memory.value['genre']}' 입니다."


# -----------------------------------
# 6. 최근 영화 검색 + 추천 전용 서브에이전트
# -----------------------------------
movie_recommender_agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=[],
    system_prompt=(
        "당신은 영화 추천 전문가입니다. "
        "입력으로 사용자의 선호 장르와 최신 웹 검색 결과가 주어집니다. "
        "검색 결과에 나온 최근 영화들 중에서 그 장르에 맞는 작품 3~5편을 골라 추천하세요. "
        "반드시 아래 원칙을 따르세요:\n"
        "1. 검색 결과에 근거해 추천하세요.\n"
        "2. 검색 결과에 없는 영화를 지어내지 마세요.\n"
        "3. 각 영화마다 추천 이유를 1문장씩 쓰세요.\n"
        "4. 마지막에 '왜 이 영화를 골랐는지'를 짧게 요약하세요.\n"
        "5. 답변은 한국어로 하세요."
    ),
)


# -----------------------------------
# 7. 검색 기반 추천 tool
# -----------------------------------
@tool
def recommend_recent_movies(runtime: ToolRuntime[UserContext]) -> str:
    """
    저장된 좋아하는 장르를 바탕으로 DuckDuckGo에서 최근 영화를 검색한 뒤 추천합니다.
    """
    assert runtime.store is not None

    namespace = ("users", runtime.context.user_id, "movie_preferences")
    memory = runtime.store.get(namespace, "favorite_genre")

    if memory is None:
        return "좋아하는 영화 장르가 저장되어 있지 않습니다. 먼저 좋아하는 장르를 알려주세요."

    genre = memory.value["genre"]

    # 최근작 중심으로 검색
    search_query = (
        f"넷플릭스영화중 {genre}장르의 영화를 찾아줘. 영화 이름하고 장르로 데이터를 가져와줘. "
    )

    try:
        search_results = ddg_tool.invoke(search_query)
    except Exception as e:
        return f"영화 검색 중 오류가 발생했습니다: {e}"

    if not search_results:
        return f"'{genre}' 장르에 대한 최근 영화 검색 결과를 찾지 못했습니다."

    
    print('덕덕고 검색 결과:',search_results)
    result = movie_recommender_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"사용자가 좋아하는 영화 장르는 '{genre}' 입니다.\n\n"
                        f"아래는 DuckDuckGo로 찾은 최근 영화 관련 검색 결과입니다.\n\n"
                        f"{search_results}\n\n"
                        f"이 자료를 바탕으로 추천해주세요."
                    ),
                }
            ]
        }
    )

    return result["messages"][-1].content

In [16]:
# -----------------------------------
# 8. 메인 에이전트
# -----------------------------------
agent = create_agent(
    model="gpt-5.4-mini",
    tools=[save_favorite_genre, load_favorite_genre, recommend_recent_movies],
    middleware=[log_tool_calls],
    store=store,
    context_schema=UserContext,
    system_prompt=(
        "당신은 사용자의 영화 취향을 기억하고 추천해주는 도우미입니다.\n"
        "- 사용자가 좋아하는 영화 장르를 말하면 save_favorite_genre를 사용하세요.\n"
        "- 사용자가 저장된 장르를 물어보면 load_favorite_genre를 사용하세요.\n"
        "- 사용자가 영화 추천을 요청하면 recommend_recent_movies를 사용하세요.\n"
        "- 답변은 한국어로 하세요."
    ),
)


# -----------------------------------
# 9. 실행 예시
# -----------------------------------
user_context = UserContext(user_id="user_001")

# 1) 장르 저장
response1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "나는 영화 액션 장르를 좋아해. 기억해줘."}
        ]
    },
    context=user_context,
)
print("=== 응답 1 ===")
print(response1["messages"][-1].content)

# 2) 장르 확인
response2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "내가 좋아하는 영화 장르가 뭐였지?"}
        ]
    },
    context=user_context,
)
print("\n=== 응답 2 ===")
print(response2["messages"][-1].content)

# 3) 최근 영화 추천
response3 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "내 취향에 맞는 최근 영화 추천해줘."}
        ]
    },
    context=user_context,
)
print("\n=== 응답 3 ===")
print(response3["messages"][-1].content)

[TOOL START]save_favorite_genre/args={'genre': 'action'}
[TOOL END]save_favorite_genre/result=좋아하는 영화 장르를 'action'로 저장했습니다.
=== 응답 1 ===
좋아요! 좋아하는 영화 장르를 **액션(action)**으로 기억해둘게요.
[TOOL START]load_favorite_genre/args={}
[TOOL END]load_favorite_genre/result=저장된 좋아하는 영화 장르는 'action' 입니다.

=== 응답 2 ===
저장된 좋아하는 영화 장르는 **action**입니다.
[TOOL START]recommend_recent_movies/args={}
덕덕고 검색 결과: [{'snippet': "이영애가 영화 '나를 찾아줘'로 다시 우리에게 돌아왔다. 그 옛날 친절한 금자씨 이후로 스크린을 떠났었는데 그때 영화가 하도 강렬해서 기억에 남아있다.", 'title': '영화 나를 찾아줘 이영애-줄거리와 결말 스포 : 네이버 블로그', 'link': 'https://m.blog.naver.com/boshukr/221747246095'}, {'snippet': '나를 찾아줘 영화를 온라인에서 무료로 HD로 시청하세요.관련 영화.', 'title': '나를 찾아줘 (2014) 영화를 온라인에서 무료로... | HDFilmCehennemi', 'link': 'https://filmcehennemi.my/ko/yeonghwa/나를-찾아줘-A210577A'}, {'snippet': '"넷플릭스에서 코미디 콘텐츠를 보여줘." "넷플릭스에서 스티븐 스필버그의 영화를 찾아줘."', 'title': '음성 제어 기능으로 넷플릭스를 이용하는 방법 | 넷플릭스 고객 센터', 'link': 'https://help.netflix.com/ko/node/111997'}, {'snippet': '캐리비안 해적 영화 찾아줘 [1].6583044. 찰갑옷과판금갑옷을 전부사용하는 삼국시

### 3) wrap_model_call : 툴 제한

In [28]:
from typing import TypedDict, Literal
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.tools import tool

#----------- wrap_tool_calling -------------#
from collections.abc import Callable

from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest
from langgraph.types import Command

In [35]:
# ---------------------------
# 1. 사용자 컨텍스트 정의
# ---------------------------
class UserContext(TypedDict):
    role: Literal['user', 'admin', 'operator']


# ---------------------------
# 2. 예시 툴 정의
# ---------------------------
@tool
def search_docs(query: str) -> str:
    '''문서를 검색합니다.'''
    return f"[SEARCH] '{query}'에 대한 검색 결과입니다."

@tool
def update_db(record_id: str, new_value: str) -> str:
    '''관리자만 사용할 수 있는 DB 수정 툴입니다.'''
    return f"[DB UPDATE] record_id={record_id}, new_value={new_value}"

@tool
def delete_file(filename: str) -> str:
    '''운영자만 사용할 수 있습니다. 파일을 삭제합니다.'''
    return f"[FILE DELETE] '{filename}' 파일을 삭제했습니다."


ALL_TOOLS = [search_docs, update_db, delete_file]


# ---------------------------
# 3. 역할별 허용 툴 맵
# ---------------------------
ROLE_TOOLS = {
    "user": [search_docs],
    "admin": [search_docs, update_db],
    "operator": [search_docs, delete_file],
}


# ---------------------------
# 4. 모델 호출 직전에 툴 제한하는 미들웨어
# ---------------------------
@wrap_model_call
def restrict_tools_by_role(request, handler):
    role = request.runtime.context.get("role", "user")
    allowed_tools = ROLE_TOOLS.get(role, [search_docs])

    allowed_tool_names = [tool_.name for tool_ in allowed_tools]
    print(f"[ROLE] {role}")
    print(f"[ALLOWED TOOLS] {allowed_tool_names}")

    # 현재 모델 호출에서만 사용할 툴을 교체
    new_request = request.override(tools=allowed_tools)
    return handler(new_request)


@wrap_tool_call
def log_tool_calls(
    request: ToolCallRequest,                                    # request: Agent가 갖고있는 정보들
    handler: Callable[[ToolCallRequest], ToolMessage | Command], # handler: Agent == "다음 실행 단계"
) -> ToolMessage | Command:
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]

    print(f"[TOOL START]{tool_name}/args={tool_args}")

    try:
        result = handler(request)

        if isinstance(result, ToolMessage):
            print(f"[TOOL END]{tool_name}/result={result.content}")
        else:
            print(f"[TOOL END]{tool_name}/result={result}")

        return result

    except Exception as e:
        print(f"[TOOL ERROR]{tool_name}/error={e}")
        raise

In [36]:
# ---------------------------
# 5. 에이전트 생성
# ---------------------------
agent = create_agent(
    model='openai:gpt-5.4-mini',
    tools=[search_docs,update_db,delete_file],
    middleware=[restrict_tools_by_role, log_tool_calls],
    context_schema=UserContext
)

In [37]:
# ---------------------------
# 6. 실행 예시
# ---------------------------

# 6-1) 일반 사용자: 검색만 가능
result = agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': '문서에서 rag와 관련된 문서를 찾아줘.'
            }
        ]
    },
    context={'role':'user'}
)

[ROLE] user
[ALLOWED TOOLS] ['search_docs']
[TOOL START]search_docs/args={'query': 'RAG 관련 문서'}
[TOOL END]search_docs/result=[SEARCH] 'RAG 관련 문서'에 대한 검색 결과입니다.
[ROLE] user
[ALLOWED TOOLS] ['search_docs']


In [39]:
print(f"{result['messages'][-1].content}")

문서에서 **RAG 관련 문서**를 찾았습니다.

검색 결과:
- **'RAG 관련 문서'에 대한 검색 결과입니다.**

현재 검색 결과에는 문서 목록이 구체적으로 표시되지 않았습니다.  
원하시면 제가 아래처럼 더 좁혀서 다시 찾아드릴 수 있어요:

- **RAG**
- **Retrieval Augmented Generation**
- **검색증강생성**
- **벡터 검색 / 임베딩 / 검색 시스템**

원하시는 기준이 있으면 말씀해 주세요.


## 2. Node-style Hooks

### 1) before_model

In [40]:
from langchain.agents.middleware import before_model
from langchain.messages import SystemMessage

In [41]:
@before_model
def force_korean(state, runtime):
    print("state: ", state)
    print(runtime)
    return {
        'messages': [
            SystemMessage(content='모든 답변은 반드시 한국어로 작성하세요.')
        ] + state['messages']
    }

In [42]:
agent = create_agent(
    model='openai:gpt-5.4-mini',
    middleware=[force_korean]
)

In [44]:
result = agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': 'Explain what RAG is.'
            }
            
        ]
    }
)

state:  {'messages': [HumanMessage(content='Explain what RAG is.', additional_kwargs={}, response_metadata={}, id='f738ac25-3b14-4a99-bb61-4a18caa097aa')]}
Runtime(context=None, store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x0000024B7E0BC5E0>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f13d319-c917-6684-8000-2badecc5104a', checkpoint_ns='force_korean.before_model:35611bb6-3370-328c-e160-55eaed7f9109', task_id='35611bb6-3370-328c-e160-55eaed7f9109', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1776741871.4666626), server_info=None)


In [45]:
print(f"{result['messages'][-1].content}")

RAG는 **Retrieval-Augmented Generation**의 약자로, 한국어로는 보통 **검색 증강 생성**이라고 합니다.

쉽게 말하면:

- **LLM(대규모 언어 모델)** 이 바로 답을 생성하는 대신,
- 먼저 **관련 문서나 데이터에서 정보를 검색(retrieval)** 한 다음,
- 그 검색된 내용을 바탕으로 **답을 생성(generation)** 하는 방식입니다.

## 왜 쓰나요?
LLM은 학습된 지식만으로 답하기 때문에:
- 최신 정보를 모를 수 있고
- 내부 사내 문서나 특정 DB 내용을 알지 못하며
- 때로는 그럴듯하지만 틀린 답을 만들 수 있습니다.

RAG는 이런 문제를 줄이기 위해,
**외부 지식 소스(문서, 위키, DB, PDF 등)** 를 참고하게 합니다.

## 동작 방식
보통 흐름은 이렇습니다:

1. 사용자가 질문
2. 시스템이 관련 문서 검색
3. 검색된 문서를 LLM에 함께 전달
4. LLM이 그 문서를 근거로 답변 생성

## 예시
질문: “우리 회사 환불 정책이 뭐야?”

- 일반 LLM: 학습 데이터에 없으면 정확히 모를 수 있음
- RAG 시스템: 회사 정책 문서를 검색해서 그 내용을 바탕으로 답변함

## 장점
- 최신 정보 반영 가능
- 사내/도메인 지식 활용 가능
- 환각(hallucination) 감소
- 답변 근거를 제시하기 쉬움

## 단점
- 검색 품질이 나쁘면 답변도 나빠짐
- 시스템이 복잡해짐
- 검색/인덱싱 비용이 듦

원하시면 제가 이어서 **“RAG와 파인튜닝의 차이”** 또는 **“RAG 시스템 구조”** 도 설명해드릴게요.
